In [1]:
# ============================================================
# 03_baselines.ipynb — Cell 1: Setup & Config
# ============================================================
# TF-IDF + Logistic Regression baselines for both targets:
#   - recommendation_status (3 classes, imbalanced)
#   - rating_band (3 ordinal classes, imbalanced)
#
# This cell only prepares the environment and shared helpers.
# No data is read or written here.
# ============================================================

from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import polars as pl
import joblib

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_fscore_support,
)

# --- Shared seed (same as Part B, for end-to-end reproducibility) ----------
SPLIT_SEED = 2026
MODEL_SEED = SPLIT_SEED  # TF-IDF is deterministic; LogReg uses this for solver

# --- Project paths ---------------------------------------------------------
PROJECT_ROOT = Path("/home/jvd/Desktop/persian-nlp-pipeline")
SPLITS_DIR   = PROJECT_ROOT / "data" / "splits"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"

TRAIN_SUPERVISED_CSV = SPLITS_DIR / "train_supervised.csv"
VALIDATION_CSV       = SPLITS_DIR / "validation.csv"
TEST_CSV             = SPLITS_DIR / "test.csv"

# --- Output paths (written by later cells) ---------------------------------
TFIDF_REC_PATH      = ARTIFACTS_DIR / "tfidf_recommendation.joblib"
TFIDF_BAND_PATH     = ARTIFACTS_DIR / "tfidf_rating_band.joblib"
BASELINE_METRICS_JSON = ARTIFACTS_DIR / "baseline_metrics.json"

# --- rating_band ordinal encoding (per split_manifest.json) ----------------
BAND_ORDINAL = {"low": 0, "medium": 1, "high": 2}

# --- Fail-fast: input files must exist -------------------------------------
for _p in (TRAIN_SUPERVISED_CSV, VALIDATION_CSV, TEST_CSV):
    assert _p.exists(), f"Missing split file: {_p}"

# ============================================================
# Shared metric helpers (reused by Transformer notebooks later)
# ============================================================

def compute_classification_metrics(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    labels: list[str] | None = None,
) -> dict:
    """
    Compute the full metric suite required by the project Definition:
      - Accuracy
      - Macro F1
      - Macro Precision, Macro Recall
      - Per-class Precision / Recall / F1
      - Balanced Accuracy
      - Confusion Matrix

    Parameters
    ----------
    y_true, y_pred : array-like of shape (n,)
        Ground-truth and predicted labels (strings or ints).
    labels : list, optional
        The canonical label order. If None, inferred from sorted union.
        Always pass this explicitly to keep the confusion matrix order stable.

    Returns
    -------
    dict with all metrics, JSON-serializable (no numpy scalars).
    """
    if labels is None:
        labels = sorted(set(y_true) | set(y_pred))

    # Core scalars
    acc    = accuracy_score(y_true, y_pred)
    bal_acc = balanced_accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average="macro", labels=labels, zero_division=0)
    macro_p, macro_r, _, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", labels=labels, zero_division=0
    )

    # Per-class metrics
    p_per, r_per, f1_per, support = precision_recall_fscore_support(
        y_true, y_pred, labels=labels, zero_division=0
    )
    per_class = {
        label: {
            "precision": float(p_per[i]),
            "recall": float(r_per[i]),
            "f1": float(f1_per[i]),
            "support": int(support[i]),
        }
        for i, label in enumerate(labels)
    }

    # Confusion matrix (rows = true, cols = predicted)
    cm = confusion_matrix(y_true, y_pred, labels=labels).tolist()

    return {
        "accuracy": float(acc),
        "balanced_accuracy": float(bal_acc),
        "macro_f1": float(macro_f1),
        "macro_precision": float(macro_p),
        "macro_recall": float(macro_r),
        "per_class": per_class,
        "confusion_matrix": cm,
        "labels_order": list(labels),
    }


def ordinal_distance_error(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    ordinal_map: dict[str, int],
) -> float:
    """
    Mean absolute ordinal distance between predicted and true labels.

    For rating_band: low=0, medium=1, high=2.
    Confusing low with high costs 2; confusing low with medium costs 1.
    This is the metric Definition requires for Task B.
    """
    true_ord = np.array([ordinal_map[v] for v in y_true])
    pred_ord = np.array([ordinal_map[v] for v in y_pred])
    return float(np.mean(np.abs(true_ord - pred_ord)))


print("Part C (baselines) environment ready.")
print(f"PROJECT_ROOT : {PROJECT_ROOT}")
print(f"SPLIT_SEED   : {SPLIT_SEED}")
print(f"MODEL_SEED   : {MODEL_SEED}")
print(f"BAND_ORDINAL : {BAND_ORDINAL}")
print()
print("Planned outputs (written by later cells):")
for p in (TFIDF_REC_PATH, TFIDF_BAND_PATH, BASELINE_METRICS_JSON):
    print(f"  {p.relative_to(PROJECT_ROOT)}")

Part C (baselines) environment ready.
PROJECT_ROOT : /home/jvd/Desktop/persian-nlp-pipeline
SPLIT_SEED   : 2026
MODEL_SEED   : 2026
BAND_ORDINAL : {'low': 0, 'medium': 1, 'high': 2}

Planned outputs (written by later cells):
  artifacts/tfidf_recommendation.joblib
  artifacts/tfidf_rating_band.joblib
  artifacts/baseline_metrics.json


In [2]:
# ============================================================
# 03_baselines.ipynb — Cell 2: Load Splits & Verify
# ============================================================
# Goal:
#   - load the three split CSVs with only the needed columns
#   - verify row counts against split_manifest.json
#   - print class distributions per split for audit
# No files are written here.
# ============================================================

# --- 1. Read split_manifest.json -------------------------------------------
with (SPLITS_DIR / "split_manifest.json").open("r", encoding="utf-8") as f:
    manifest = json.load(f)

manifest_counts = manifest["counts"]
print("Manifest row counts:")
print(f"  train_supervised       : {manifest_counts['train_supervised']:,}")
print(f"  validation             : {manifest_counts['validation']:,}")
print(f"  test                   : {manifest_counts['test']:,}")
print()

# --- 2. Load the three splits ---------------------------------------------
# Only the columns needed for TF-IDF baselines. id and product_id are kept
# for downstream audit; the target columns are kept as-is (nulls preserved).
USECOLS = [
    "id",
    "body_normalized",
    "recommendation_status",
    "rating_band",
    "is_valid_rating_band",
    "product_id",
]

NULL_TOKENS = ["", "nan", "NaN", "null", "None"]

def load_split(path: Path) -> pl.DataFrame:
    return pl.read_csv(
        path,
        separator=",",
        has_header=True,
        null_values=NULL_TOKENS,
        infer_schema_length=10_000,
        try_parse_dates=False,
    ).select(USECOLS)

train = load_split(TRAIN_SUPERVISED_CSV)
val   = load_split(VALIDATION_CSV)
test  = load_split(TEST_CSV)

# --- 3. Verify row counts against the manifest -----------------------------
for name, df, expected in [
    ("train", train, manifest_counts["train_supervised"]),
    ("validation", val, manifest_counts["validation"]),
    ("test", test, manifest_counts["test"]),
]:
    actual = df.height
    status = "OK" if actual == expected else f"MISMATCH (expected {expected:,})"
    print(f"Loaded {name:<12} : {actual:>10,} rows  [{status}]")
    assert actual == expected, (
        f"{name} row count mismatch: got {actual}, expected {expected}. "
        f"Was the split file modified after B5?"
    )

print()
print("All three splits match split_manifest.json.")
print()

# --- 4. Distribution of recommendation_status per split (supervised only) -
def rec_dist(df: pl.DataFrame) -> pl.DataFrame:
    return (
        df
        .filter(pl.col("recommendation_status").is_not_null())
        .group_by("recommendation_status")
        .agg(pl.len().alias("n"))
        .with_columns(
            (pl.col("n") / pl.col("n").sum() * 100).round(2).alias("pct")
        )
        .sort("n", descending=True)
    )

print("recommendation_status distribution (supervised rows only):")
for name, df in [("train", train), ("validation", val), ("test", test)]:
    print(f"\n  {name}:")
    print(rec_dist(df))
print()

# --- 5. Distribution of rating_band per split (valid rows only) -----------
def band_dist(df: pl.DataFrame) -> pl.DataFrame:
    return (
        df
        .filter(pl.col("is_valid_rating_band") & pl.col("rating_band").is_not_null())
        .group_by("rating_band")
        .agg(pl.len().alias("n"))
        .with_columns(
            (pl.col("n") / pl.col("n").sum() * 100).round(2).alias("pct")
        )
        .sort("rating_band")
    )

print("rating_band distribution (valid rows only):")
for name, df in [("train", train), ("validation", val), ("test", test)]:
    print(f"\n  {name}:")
    print(band_dist(df))
print()

# --- 6. Quick shape summary ------------------------------------------------
print("Quick summary:")
print(f"  train      : {train.height:>10,} rows  | "
      f"supervised rec={train.filter(pl.col('recommendation_status').is_not_null()).height:>10,}  | "
      f"valid band={train.filter(pl.col('is_valid_rating_band')).height:>10,}")
print(f"  validation : {val.height:>10,} rows  | "
      f"supervised rec={val.filter(pl.col('recommendation_status').is_not_null()).height:>10,}  | "
      f"valid band={val.filter(pl.col('is_valid_rating_band')).height:>10,}")
print(f"  test       : {test.height:>10,} rows  | "
      f"supervised rec={test.filter(pl.col('recommendation_status').is_not_null()).height:>10,}  | "
      f"valid band={test.filter(pl.col('is_valid_rating_band')).height:>10,}")

Manifest row counts:
  train_supervised       : 3,578,692
  validation             : 913,570
  test                   : 919,841

Loaded train        :  3,578,692 rows  [OK]
Loaded validation   :    913,570 rows  [OK]
Loaded test         :    919,841 rows  [OK]

All three splits match split_manifest.json.

recommendation_status distribution (supervised rows only):

  train:
shape: (3, 3)
┌───────────────────────┬─────────┬───────┐
│ recommendation_status ┆ n       ┆ pct   │
│ ---                   ┆ ---     ┆ ---   │
│ str                   ┆ u32     ┆ f64   │
╞═══════════════════════╪═════════╪═══════╡
│ recommended           ┆ 2336829 ┆ 79.19 │
│ no_idea               ┆ 332579  ┆ 11.27 │
│ not_recommended       ┆ 281521  ┆ 9.54  │
└───────────────────────┴─────────┴───────┘

  validation:
shape: (3, 3)
┌───────────────────────┬────────┬───────┐
│ recommendation_status ┆ n      ┆ pct   │
│ ---                   ┆ ---    ┆ ---   │
│ str                   ┆ u32    ┆ f64   │
╞════════════

In [3]:
# ============================================================
# 03_baselines.ipynb — Cell 3 (LIGHTWEIGHT): TF-IDF Vectorizer
# ============================================================
# Lightweight configuration: word-level TF-IDF only, 150K features,
# min_df=5. This keeps peak RAM around 4-5 GB for 3.5M documents,
# which fits comfortably on a 14 GB machine.
#
# Char-level n-grams are intentionally dropped here. They can be
# added in Phase 2 as an extension with tuned hyperparameters
# (max_features<=50K, min_df>=20) once memory budget is known.
#
# No files are written. The vectorizer is saved in cell 7.
# ============================================================

import time
import numpy as np

# --- 1. Prepare the target texts (fill nulls with empty strings) ---------
train_texts = train["body_normalized"].fill_null("").to_list()
val_texts   = val["body_normalized"].fill_null("").to_list()
test_texts  = test["body_normalized"].fill_null("").to_list()

print("Text corpus sizes:")
print(f"  train : {len(train_texts):>10,}")
print(f"  val   : {len(val_texts):>10,}")
print(f"  test  : {len(test_texts):>10,}")
print()

# --- 2. Build the word-level TF-IDF --------------------------------------
print("Fitting word-level TF-IDF (1-2 grams, max_features=150K, min_df=5)...")
t0 = time.time()
word_vec = TfidfVectorizer(
    analyzer="word",
    ngram_range=(1, 2),
    min_df=5,
    max_df=0.95,
    max_features=150_000,
    sublinear_tf=True,
    lowercase=False,
    strip_accents=None,
    dtype=np.float32,
)
X_train = word_vec.fit_transform(train_texts)
X_val   = word_vec.transform(val_texts)
X_test  = word_vec.transform(test_texts)
t_word = time.time() - t0
print(f"  fit+transform in {t_word:.1f}s")
print(f"  vocabulary size : {len(word_vec.vocabulary_):,}")
print()

# --- 3. Report shape and nnz --------------------------------------------
print("Feature matrices (word-only):")
print(f"  X_train : {X_train.shape}  nnz={X_train.nnz:,}")
print(f"  X_val   : {X_val.shape}    nnz={X_val.nnz:,}")
print(f"  X_test  : {X_test.shape}   nnz={X_test.nnz:,}")
print()

# --- 4. Sanity checks ---------------------------------------------------
assert X_train.shape[1] == X_val.shape[1] == X_test.shape[1], \
    "feature dimension mismatch across splits"
assert np.isfinite(X_train.data).all(), "NaN/Inf in X_train"
assert np.isfinite(X_val.data).all(),   "NaN/Inf in X_val"
assert np.isfinite(X_test.data).all(),  "NaN/Inf in X_test"
assert len(word_vec.vocabulary_) > 0

print("Sanity checks passed.")
print(f"  total features : {X_train.shape[1]:,}")
print(f"  total fit time : {t_word:.1f}s")
print()

# --- 5. Estimated RAM of feature matrices (int32 indices + float32 values)
total_nnz = X_train.nnz + X_val.nnz + X_test.nnz
approx_ram_gb = total_nnz * (4 + 4) / 1024**3
print(f"Approximate RAM for feature matrices : {approx_ram_gb:.2f} GB "
      f"(nnz={total_nnz:,})")

Text corpus sizes:
  train :  3,578,692
  val   :    913,570
  test  :    919,841

Fitting word-level TF-IDF (1-2 grams, max_features=150K, min_df=5)...
  fit+transform in 74.5s
  vocabulary size : 150,000

Feature matrices (word-only):
  X_train : (3578692, 150000)  nnz=54,886,707
  X_val   : (913570, 150000)    nnz=14,019,832
  X_test  : (919841, 150000)   nnz=14,093,326

Sanity checks passed.
  total features : 150,000
  total fit time : 74.5s

Approximate RAM for feature matrices : 0.62 GB (nnz=82,999,865)
